<center>
    <p>105 Theoretical Neuroscience I</p>
    <h1></h1>
    <h1>Lecture 15:</h1>
    <h1>Decoding populations of neurons</h1>
    <p>----</p>
    <p>Prof. Jochen Braun Ph.D.</p>
    <p>Institute of Biology</p>
    <p>Otto-von-Guericke University Magdeburg</p>
    <p>----</p>
    <p>Textbook:</p>
    <p>Peter Dayan & Larry Abbott (2001) Theoretical Neuroscience, MIT Press.</p>
</center>

# **Abstract:**

Today we ask how Bayesian inference could be implemented with neurons? After recapitulating the **maximum likelihood (ML)** formalism and developing a graphial intuition for its workings, we take non-uniform priors into account and extend the formalism to **maximum a posteriori (MAP)** likelihood.

After discovering that maximum likelihood estimates involve **response-weighted averaging over heterogeneous neurons**, we will be ready to tackle neural implementation.

Following Jazayeri \& Movshon (2006), we show that maximul likelihood can be obtained weighting the observed **sensory responses** with the logarithm of the expected responses to different possible stimuli. In order to weigh observed sensory responses differently for different possible stimuli, we use a feedforward projection to **likelihood neurons**.

We conclude that maximum likelihood decoding of neural populations can be readily achieved with feedforward projections.



# **Overview:**



1. **Maximum likelihood inference, ML**

2. **Maximum a posteriori inference, MAP**

3. **Computing log-likelihood with feedforward projections**

4. **Detecting, discriminating and identifying with feedforward projections**

**Credits:**

Anthony Movshon, New York University

[Jazayeri \& Movshon (2006) Nature Neuroscience, 9: 690-696](https://www.nature.com/articles/nn1691)



# **1. Maximum likelihood inference (ML)**


- A 'hetereogeneous' population comprises neurons $k$ with a variety of tuning (stimulus preferences $\theta_k^\mathit{pref}$).

- Assume that an unknown stimulus $\theta$ evokes a set of responses $x_k$ from such population.

- To interpret these responses, we need to known the specific preference of each neuron $k$. Specifically, we need to know the probability $P_k(x_k|\theta)$ of response $x_k$, given any possible stimuli $\theta$.

- The likelihood $L_k(\theta|x_k)$ of a stimulus $\theta$, given response $x_k$, is proportional to $P_k(x_k|\theta)$, as long as the marginal stimulus probability is uniform, $P(\theta) = \mathit{const}$:

$\begin{eqnarray}
\qquad\qquad L_k(\theta|x_k) = \frac{P_k(x_k,\theta)}{P(x_k)} = \frac{P(\theta)}{P_k(x_k)}\frac{P_k(x_k,\theta)}{P(\theta)}= \mathit{const} \cdot \frac{P_k(x_k,\theta)}{P_k(\theta)} = \mathit{const} \cdot P_k(x_k|\theta)
\end{eqnarray}$





- The joint likelihood $L(\theta|\{x_1, \ldots , x_n\} )$ of a stimulus $\theta$, given multiple independent responses $x_1, \ldots, x_n$ is

$\begin{eqnarray}
\qquad\qquad L(\theta|x_1, \dots , x_n )= L_1(\theta|x_1) \, L_1(\theta|x_2 ) \, \ldots \,  L_n(\theta|x_n )
\end{eqnarray}$

- The **maximum likelihood estimate** $\theta_{est}$ -- most likely stimulus to have evoked the observed response -- is obtained by solving

$\begin{eqnarray}
\qquad\qquad \frac{\partial}{\partial\theta} \prod_k\,L_k(\theta| x_k)= 0  \qquad \Leftrightarrow \qquad \frac{\partial}{\partial\theta} \sum_k \,  \ln L_k(\theta| x_k) = 0
\end{eqnarray}$

(Slide-1)


# Likelihood of stimulus $\theta$, given one response $x$

We seek to develop an intuition for the ML computation and its dependence on response variability and tuning curve. For analytical convenience, we assume Poisson variability and Gaussian tuning:

$\begin{eqnarray}
L(\theta|x) = \frac{ [r(\theta)]^x \, e^{-r(\theta)}}{x!},\qquad\qquad r(\theta)= r_{max} \, \exp\left[-\frac{ (\theta-\theta_{pref})^{2}}{2\sigma^{2}} \right]
\end{eqnarray}$

The log likelihood/probability depends on $\theta$ and $x$ in a way that reflects both response variability and tuning curve. After working out the partial derivatives

$\begin{eqnarray}
\frac{\partial \ln L(\theta|x)}{\partial r}  =  \frac{x - r(\theta)}{r(\theta)},\qquad\qquad\frac{\partial r(\theta)}{\partial\theta}  = -\frac{\theta-\theta_{pref}}{\sigma^2} \, r(\theta)
\end{eqnarray}$

we see that an observed response $x$ contributes to the ML condition the following expression

$\begin{eqnarray}
\frac{\partial \ln L(\theta|x)}{\partial\theta}  &=& \frac{\partial \ln L(\theta|x)}{\partial r} \,  \frac{\partial r(\theta)}{\partial\theta} = - \frac{\left[x - r(\theta)\right]\,\left[\theta-\theta_{pref}\right]}{\sigma^2} \quad \stackrel{!}{=} 0
\end{eqnarray}$

(Slide-2)


In [ ]:
from inspect import signature
# @title One contribution to ML condition {"vertical-output":true,"display-mode":"form"}
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact
from scipy.stats import poisson
from scipy.stats import norm
from matplotlib.colors import LinearSegmentedColormap

def SignedAngleDiff( theta_i, theta_pref ):
    # angle difference between preferred theta_pref and possible theta_i, on circular range
    delta_i = theta_i - theta_pref;
    delta_i[delta_i >  np.pi] = delta_i[delta_i >  np.pi] - 2*np.pi;
    delta_i[delta_i < -np.pi] = delta_i[delta_i < -np.pi] + 2*np.pi;

    return delta_i

def TuningCurvePrime( theta_pref, theta_i, sigma, rmax ):

    # initialize array of population responses
    tuning_i = np.zeros(len(theta_i));

    # angle difference between preferred theta_pref and possible theta_i, on circular range
    delta_i = SignedAngleDiff( theta_i, theta_pref )

    # tuning or expected response
    tuning_i = rmax * np.exp( - delta_i**2 / (2*sigma**2) )

    # derivative w.r.t. theta_i
    tuning_i_prime = - delta_i / sigma**2 * tuning_i

    return tuning_i, tuning_i_prime, delta_i

def LogVariabilityPrime( x, tuning_i ):

    lnL_i = x * np.log(tuning_i) - tuning_i - x * np.log(x) + x;

    lnL_i_prime = (x - tuning_i) / tuning_i;

    return lnL_i, lnL_i_prime

def plot_one_contribution_to_ML(Sigma=0.7, Theta_pref=np.pi/4, toggle=0):
    """(Slide-3)"""

    Rmax = 10

    Nresolution = 200

    # range of possible stimuli
    Theta_i = np.linspace(-np.pi, np.pi, Nresolution, endpoint=False );

    # randomly generate an observed response
    x_obs = Rmax * (np.random.rand() + 0.5);

    # expected response of a specific neuron
    (tun_i, tun_i_prime, delta_i) = TuningCurvePrime( Theta_pref, Theta_i, Sigma, Rmax );
    partial_tun_i = tun_i_prime / tun_i;

    # log likelihood of observed response, given expected response
    (lnL_i, lnL_i_prime) = LogVariabilityPrime( x_obs, tun_i );
    partial_lnL_i = lnL_i_prime * tun_i;

    # ML condition
    lnL_i_ml = tun_i_prime * lnL_i_prime;

    # Plot results
    fig, axs = plt.subplots( 1,3, figsize=(9,3))

    YLim = [-1.5*Rmax, +1.5*Rmax];
    XLim = [Theta_pref-np.pi/2, Theta_pref+np.pi/2];

    axs[0].plot( Theta_i, tun_i, 'b-', linewidth=2, label=r"$r(\theta)$")
    axs[0].plot( Theta_i, tun_i_prime, 'r-', linewidth=2, label=r"$\partial r(\theta)/\partial \theta$")
    axs[0].plot( [Theta_pref,Theta_pref], YLim, 'r:', linewidth=2, label=r"$\theta_\mathit{pref}$" )
    axs[0].plot( XLim, [x_obs, x_obs], 'k:', linewidth=2, label=r"$x_\mathit{obs}$")
    axs[0].plot( XLim, [0,0], 'k-', linewidth=0.5)

    axs[0].set_xlim(XLim)
    axs[0].legend(loc='lower left')
    axs[0].set_xlabel('stimulus ' r"$\theta$")
    axs[0].set_ylim(YLim)
    axs[0].set_box_aspect(1/1)
    axs[0].set_title('Tuning', fontsize=14, color='k');

    axs[1].plot( Theta_i, lnL_i, 'b-', linewidth=2, label=r"$\ln L(\theta|x)$")
    axs[1].plot( Theta_i, lnL_i_prime, 'r-', linewidth=2, label=r"$\partial \ln L(\theta|x)/\partial r$")
    axs[1].plot( [Theta_pref,Theta_pref], YLim, 'r:', linewidth=2, label=r"$\theta_\mathit{pref}$" )
    axs[1].plot( XLim, [0,0], 'k-', linewidth=0.5 )

    axs[1].set_xlim(XLim)
    axs[1].legend(loc='lower left')
    axs[1].set_xlabel('stimulus ' r"$\theta$")
    axs[1].set_ylim(YLim)
    axs[1].set_box_aspect(1/1)
    axs[1].set_title('Variability', fontsize=14, color='k');

    axs[2].plot( Theta_i, lnL_i_ml, 'r-', linewidth=2, label=r"$\partial \ln L(\theta|x) / \partial \theta$")
    axs[2].plot( [Theta_pref,Theta_pref], YLim, 'r:', linewidth=2, label=r"$\theta_\mathit{pref}$")
    axs[2].plot( XLim, [0,0], 'k-', linewidth=0.5)

    axs[2].set_xlim(XLim)
    axs[2].legend(loc='lower left')
    axs[2].set_xlabel('stimulus ' r"$\theta$")
    axs[2].set_ylim(YLim)
    axs[2].set_box_aspect(1/1)
    axs[2].set_title('Product', fontsize=14, color='k');

    plt.tight_layout()
    plt.show()

# Creating sliders for concentrations and charge
interact(plot_one_contribution_to_ML, Sigma = (0.2, 1.5, 0.1), Theta_pref = (-np.pi/2, +np.pi/2, 0.1), toggle = (0, 1, 1) )


interactive(children=(FloatSlider(value=0.7, description='Sigma', max=1.5, min=0.2), FloatSlider(value=0.78539…

<function __main__.plot_one_contribution_to_ML(Sigma=0.7, Theta_pref=0.7853981633974483, toggle=0)>

# Likelihood of stimulus $\theta$, given multiple responses $x_1, \ldots, x_n$

Maximize likelihood / probability

$\begin{eqnarray}
L(\theta|\{x_1, \ldots, x_n\}) = \prod_{k=1}^n \, L_k(\theta|x_k)
\end{eqnarray}$

by solving for extremum of

$\begin{eqnarray}
\frac{\partial}{\partial\theta}  \ln L(\theta|\{x_1, \ldots, x_n\}) &=& \frac{\partial}{\partial\theta} \sum_{k=1}^n \ln L_k(\theta|x_k) =  \sum_{k=1}^n \frac{\partial}{\partial\theta} \ln  L_k(\theta|x_k) =
\\
\\
&=& - \sum_{k=1}^n \frac{\left[x_k - r_k(\theta)\right]\,\left[\theta-\theta_k^{pref}\right]}{\sigma^2}  \quad \stackrel{!}{=} \quad 0
\end{eqnarray}$

(Slide-4)

In [ ]:
from inspect import signature
# @title Multiple contributions to ML condition {"vertical-output":true,"display-mode":"form"}
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact
from scipy.stats import poisson
from scipy.stats import norm
from matplotlib.colors import LinearSegmentedColormap

def PoissonVariability( tuning_k ):

    x_k = np.random.poisson(tuning_k)

    return x_k

def TuningCurvePrime( theta_pref, theta_i, sigma, rmax ):

    # angle difference between preferred theta_pref and possible theta_i, on circular range
    delta_i = SignedAngleDiff( theta_i, theta_pref )

    # tuning or expected response
    tuning_i = rmax * np.exp( - delta_i**2 / (2*sigma**2) )

    # derivative w.r.t. theta_i
    tuning_i_prime = - delta_i / sigma**2 * tuning_i

    return tuning_i, tuning_i_prime, delta_i

def LogVariabilityPrime( x, tuning_i ):

    if x>0:

        lnL_i = x * np.log(tuning_i) - tuning_i - x * np.log(x) + x;

        lnL_i_prime = (x - tuning_i) / tuning_i;

    else:
        lnL_i = -tuning_i
        lnL_i_prime = np.full_like(tuning_i, 1)

    return lnL_i, lnL_i_prime

def plot_multiple_contributions_to_ML(Sigma=0.7, Nneuron= 20, toggle=0):
    """(Slide-5)"""

    Rmax = 10

    Nresolution = 200

    Theta_true = 0;

    # range of possible stimuli
    Theta_i = np.linspace(-np.pi, np.pi, Nresolution, endpoint=False );

    # preferred stimuli of neurons
    Theta_k = np.linspace(-np.pi, np.pi, Nneuron, endpoint=False );

    # generate expected and noisy responses evoked by the true stimulus
    (r_k, r_k_prime, delta_k) = TuningCurvePrime( Theta_k, Theta_true, Sigma, Rmax )
    x_k = PoissonVariability( r_k );

    # expected responses evoked by different possible simuli
    tun_ki = np.zeros( (Nneuron, Nresolution) )
    tun_ki_prime = np.zeros( (Nneuron, Nresolution) )

    for k in range( Nneuron ):
        (r_i, r_i_prime, delta_i) = TuningCurvePrime( Theta_k[k], Theta_i, Sigma, Rmax )
        tun_ki[k,:] = r_i;
        tun_ki_prime[k,:] = r_i_prime;

    # partial derivative of variability
    lnL_ki = np.zeros( (Nneuron, Nresolution) )
    lnL_ki_prime = np.zeros( (Nneuron, Nresolution) )

    for k in range( Nneuron ):
        (lnL_i, lnL_i_prime) = LogVariabilityPrime( x_k[k], tun_ki[k,:] )
        lnL_ki[k,:] = lnL_i;
        lnL_ki_prime[k,:] = lnL_i_prime;

    # ML condition
    lnL_ki_ml = tun_ki_prime * lnL_ki_prime;
    lnL_i_ml  = np.sum( lnL_ki_ml, axis=0 )


    # Plot results
    fig, axs = plt.subplots( 1,3, figsize=(9,3))

    XLim = [-np.pi/4, +np.pi/4];
    YLim = [-1.5*Rmax, +1.5*Rmax];

    for k in range( Nneuron ):
        if k == 0:
            axs[0].plot( Theta_i, tun_ki_prime[k,:], '-', linewidth=0.5, label=r"$\partial r_k(\theta)/\partial \theta$")
        else:
            axs[0].plot( Theta_i, tun_ki_prime[k,:], '-', linewidth=0.5 )
    axs[0].plot( [Theta_true,Theta_true], YLim, 'r:', linewidth=2, label=r"$\theta_\mathit{true}$" )
    axs[0].plot( XLim, [0,0], 'k-', linewidth=0.5)

    axs[0].set_xlim(XLim)
    axs[0].legend(loc='lower left')
    axs[0].set_xlabel('stimulus ' r"$\theta$")
    axs[0].set_ylim(YLim)
    axs[0].set_box_aspect(1/1)
    axs[0].set_title(r'Tuning', fontsize=14, color='k');

    for k in range( Nneuron ):
        if k == 0:
            axs[1].plot( Theta_i, lnL_ki_prime[k,:], '-', linewidth=0.5, label=r"$\partial \ln L_k(\theta|x_k)/\partial r_k$")
        else:
            axs[1].plot( Theta_i, lnL_ki_prime[k,:], '-', linewidth=0.5 )
    axs[1].plot( [Theta_true,Theta_true], YLim, 'r:', linewidth=2, label=r"$\theta_\mathit{true}$" )
    axs[1].plot( XLim, [0,0], 'k-', linewidth=0.5 )

    axs[1].set_xlim(XLim)
    axs[1].legend(loc='lower left')
    axs[1].set_xlabel('stimulus ' r"$\theta$")
    axs[1].set_ylim(YLim)
    axs[1].set_box_aspect(1/1)
    axs[1].set_title('Variability', fontsize=14, color='k');

    for k in range( Nneuron ):
        if k == 0:
            axs[2].plot( Theta_i, lnL_ki_ml[k,:], '-', linewidth=0.5, label=r"$\partial\ln L_k(\theta|x_k) / \partial\theta$" )
        else:
            axs[2].plot( Theta_i, lnL_ki_ml[k,:], '-', linewidth=0.5 )
    axs[2].plot( Theta_i, lnL_i_ml, 'r-', linewidth=2, label='sum of contributions' )
    axs[2].plot( [Theta_true,Theta_true], YLim, 'r:', linewidth=2, label=r"$\theta_\mathit{true}$")
    axs[2].plot( XLim, [0,0], 'k-', linewidth=0.5)

    axs[2].set_xlim(XLim)
    axs[2].legend(loc='lower left')
    axs[2].set_xlabel('stimulus ' r"$\theta$")
    axs[2].set_ylim(YLim)
    axs[2].set_box_aspect(1/1)
    axs[2].set_title('Product', fontsize=14, color='k');

    plt.tight_layout()
    plt.show()

# Creating sliders for concentrations and charge
interact(plot_multiple_contributions_to_ML, Sigma = (0.2, 1.5, 0.1), Nneuron = (10,100,10), toggle = (0, 1, 1) )


interactive(children=(FloatSlider(value=0.7, description='Sigma', max=1.5, min=0.2), IntSlider(value=20, descr…

<function __main__.plot_multiple_contributions_to_ML(Sigma=0.7, Nneuron=20, toggle=0)>

# Summary maximum likelihood inference


We have considered a population of neurons with different preferences $\theta_1^{pref}$, $\theta_2^{pref}$, $\theta_3^{pref}$, $\ldots$

<br>

-  Zero-crossing of $\partial_\theta \ln L_k(\theta|x_k)$ represents the single-neuron ML estimate of stimulus $\theta$.

<br>

- Zero-crossing of $\partial_\theta \ln L_k(\theta|x_{1:n})$ represents combined ML estimate of stimulus $\theta$.

$\begin{eqnarray}
\qquad\qquad \partial_\theta \ln L(\theta|x_{1:n}) = \sum_k \partial_\theta \ln L_k(\theta|x_k) = - \sum_{k=1}^n \frac{\left[x_k - r_k(\theta)\right]\,\left[\theta-\theta_k^{pref}\right]}{\sigma^2}
\end{eqnarray}$

<br>

- We have illustrated graphically the solution of the ML condition for multiple responses.

- We have also discovered a possibility for solving it with a **response-weighted average** over neurons.

(Slide-6)


# **2. Maximum a posteriori inference (MAP)**


Inference is 'statistically efficient' when it is as accurate as possible, given the available information and noise.  Optimal inference maximizes $L(\theta|x)$, the likelihood of stimulus $\theta$, given observed response $x$.

<br>

- Maximum **a posteriori** inference (MAP) maximizes $L(\theta|x)$, while also taking into account a **non-uniform** **marginal probability** of $\theta$.  In this context, the marginal probability is also called the **a priori** probability of stimuli $\theta$.

<br>

- In contrast, maximum likelihood inference (**ML**) maximizes $P(x|\theta)$, which is only equivalent to maximizing $L(\theta|x)$ when the **marginal probability** of different stimuli $\theta$ is **uniform**.

<br>



- The attributes **prior (a priori)** and **posterior (a posteriori)** specify the order with respect to some additional information, such as observed responses $x$.

**Prior** probability $P(\theta)$ $\quad\rightarrow\quad$

additional information (observed responses $x$) $\quad\rightarrow\quad$

**Posterior** likelikood $L(\theta|x)$ $\quad\rightarrow\quad$

yet more information (observed responses $x'$) $\quad\rightarrow\quad$

**Posterior** likelikood $L(\theta|x,x')$ $\quad\rightarrow\quad$

and so on

(Slide-7)


# ML and MAP formalisms compared


In the context of neuronal responses, both ML and MAP require extensive *prior knowledge* about the neurons, specifically, the conditional **response probability**, given a stimulus.  

$\begin{eqnarray}
\qquad\qquad\qquad P(x|\theta)
\end{eqnarray}$


MAP additionally requires the **prior probability** of different stimuli.

$\begin{eqnarray}
\qquad\qquad\qquad P(\theta)
\end{eqnarray}$

The conditional **stimulus likelihood** combines the conditional **response probability** and **prior probability** $P(\theta)$:

$\begin{eqnarray}
\qquad L(\theta|x)\quad =  \quad \frac{P(x|\theta) \, P(\theta)}{P(x)} \quad \approx \quad P(x|\theta) \, P(\theta)
\end{eqnarray}$

The logarithmic form

$\begin{eqnarray}
\ln L(\theta|x)=  \ln P(x|\theta) + \ln P(\theta) - \ln P(x)
\end{eqnarray}$

implies

$\begin{eqnarray}
\frac{\partial}{\partial \theta}  \ln L(\theta|x)= \frac{\partial}{\partial \theta}  \ln P(x|\theta), \qquad\qquad\qquad
\qquad  \textit{when} \quad P(\theta)=\mathit{const}
\end{eqnarray}$

and

$\begin{eqnarray}
\frac{\partial}{\partial \theta} \ln L(\theta|x) = \frac{\partial}{\partial \theta}  \ln P(x|s) + \frac{\partial}{\partial \theta}  \ln P(\theta),
\qquad\textit{when}\quad P(\theta)\neq\mathit{const}
\end{eqnarray}$

(Slide-8)

# Posterior joint likelihood


Given independent responses

$\begin{eqnarray}
\qquad\qquad\qquad x_{1:n} \equiv \{x_1, x_2, \ldots , x_n\},
\end{eqnarray}$

the joint likelihood of $\theta$ is the product of individual likelihoods

$\begin{eqnarray}
\qquad L(\theta|x_{1:n})  \quad=\quad \prod_{k=1}^n \, L_k(\theta|x_k) \quad\propto\quad P(\theta) \, \prod_{k=1}^n \, P( x_k | \theta )
\end{eqnarray}$

with ``MAP condition''

$\begin{eqnarray}
\qquad 0 \quad\stackrel{!}{=}\quad \frac{\partial}{\partial \theta} \ln L_i(\theta|x_{1:n})  = \frac{\partial}{\partial \theta} \ln P(\theta) + \sum_{k=1}^n \, \frac{\partial}{\partial \theta} \ln P_k( x_k | \theta )
\end{eqnarray}$

(Slide-9)


# Assumptions for analytical convenience

**Gaussian prior:**

$\begin{eqnarray}
P(s) = \frac{1}{\sqrt{2\pi} \sigma_\mathit{prior}} \, \exp\left[{-\frac{(\theta-\theta_\mathit{prior})^2}{2\sigma_\mathit{prior}^2}}\right] \qquad \Rightarrow \qquad \frac{\partial}{\partial \theta} \ln P(\theta) = -\frac{\theta-\theta_\mathit{prior}}{\sigma^2_\mathit{prior}}
\end{eqnarray}$

<br>


**Poisson variability**

$\begin{eqnarray}
P_k(x_k|\theta) = \frac{\left[r_i(\theta)\right]^x e^{-r_k(\theta)}}{x_k!} \qquad \Rightarrow \qquad\frac{\partial \ln P_k(\theta|x_k)}{\partial r_k}  =  \frac{x_k - r_k(\theta)}{r_k(\theta)}
\end{eqnarray}$

<br>

**Gaussian tuning**

$\begin{eqnarray}
r_k(\theta)= r_\mathit{max} \, \exp\left[-\frac{ (\theta-\theta_k^\mathit{pref})^{2}}{2\sigma_\mathit{tuning}^{2} }\right] \qquad \Rightarrow \qquad \frac{\partial r_k}{\partial \theta}  = -\frac{\theta-\theta_k^\mathit{pref}}{\sigma_\mathit{tuning}^2} \, r_k(\theta)
\end{eqnarray}$

(Slide-10)



# Simplifying the MAP condition

With our analytically convenient assumptions

$\begin{eqnarray}
\frac{\partial \ln P(\theta))}{\partial \theta}  &=& -\frac{\theta-\theta_\mathit{prior}}{\sigma^2_\mathit{prior}}
\end{eqnarray}$

<br>

$\begin{eqnarray}
\frac{\partial \ln P(x_k|r_k)}{\partial r_k} \,  \frac{\partial r_k}{\partial \theta} &=& - \frac{\left[x_k - r_k(\theta)\right]\,\left[\theta-\theta_k^\mathit{pref}\right]}{\sigma_{tuning}^2}
\end{eqnarray}$

<br>

the MAP condition simplifies to

$\begin{eqnarray}
0 &\stackrel{!}{=}&  -\frac{\theta - \theta_{prior}}{\sigma_{prior}^2} - \sum_{k=1}^n \,  \frac{\left[x_k - r_k(\theta)\right]\,\left[\theta-\theta_k^\mathit{pref}\right]}{\sigma_{tuning}^2}  
\end{eqnarray}$

<br>

(Slide-11)


In [ ]:
from inspect import signature
# @title Multiple contributions to MAP condition {"vertical-output":true,"display-mode":"form"}
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact
from scipy.stats import poisson
from scipy.stats import norm
from matplotlib.colors import LinearSegmentedColormap

def plot_multiple_contributions_to_MAP(Sigma_prior=0.4, Sigma_tun=1.0, Nneuron= 20, toggle=0):
    """(Slide-12)"""

    Rmax = 10

    Nresolution = 200

    Theta_true = 0;

    Theta_prior = -np.pi/4;

    # range of possible stimuli
    Theta_i = np.linspace(-np.pi, np.pi, Nresolution, endpoint=False );

    # preferred stimuli of neurons
    Theta_k = np.linspace(-np.pi, np.pi, Nneuron, endpoint=False );

    # generate expected and noisy responses evoked by the true stimulus
    (r_k, r_k_prime, delta_k) = TuningCurvePrime( Theta_k, Theta_true, Sigma_tun, Rmax )
    x_k = PoissonVariability( r_k );

    # expected responses evoked by different possible simuli
    tun_ki = np.zeros( (Nneuron, Nresolution) )
    tun_ki_prime = np.zeros( (Nneuron, Nresolution) )

    for k in range( Nneuron ):
        (r_i, r_i_prime, delta_i) = TuningCurvePrime( Theta_k[k], Theta_i, Sigma_tun, Rmax )
        tun_ki[k,:] = r_i;
        tun_ki_prime[k,:] = r_i_prime;

    # partial derivative of variability
    lnL_ki = np.zeros( (Nneuron, Nresolution) )
    lnL_ki_prime = np.zeros( (Nneuron, Nresolution) )

    for k in range( Nneuron ):
        (lnL_i, lnL_i_prime) = LogVariabilityPrime( x_k[k], tun_ki[k,:] )
        lnL_ki[k,:] = lnL_i;
        lnL_ki_prime[k,:] = lnL_i_prime;

    # ML condition
    lnL_ki_ML = tun_ki_prime * lnL_ki_prime;
    lnL_i_ML  = np.sum( lnL_ki_ML, axis=0 )

    # prior distribution
    P_i_prior = norm.pdf( Theta_i, Theta_prior, Sigma_prior );
    lnP_i_prior = - SignedAngleDiff( Theta_i, Theta_prior) / Sigma_prior**2;

    # MAP condition
    lnL_i_MAP = lnL_i_ML + lnP_i_prior;

    # Plot results
    fig, axs = plt.subplots( 1,3, figsize=(9,3))

    XLim = [-2*np.pi/4, +np.pi/4];
    YLim = [-1.5*Rmax, +1.5*Rmax];


    axs[0].plot( Theta_i, P_i_prior, 'b-', linewidth=2, label=r"$P_\mathit{prior}(\theta)$")
    axs[0].plot( Theta_i, lnP_i_prior, 'm-', linewidth=2, label=r"$-\ln P_\mathit{prior}(\theta)$")
    axs[0].plot( [Theta_prior,Theta_prior], [-1,+1], 'm:', linewidth=2, label=r"$\theta_\mathit{prior}$" )
    axs[0].plot( [Theta_true,Theta_true], [-1,+1], 'r:', linewidth=2, label=r"$\theta_\mathit{true}$" )
    axs[0].plot( XLim, [0,0], 'k-', linewidth=0.5)

    axs[0].set_xlim(XLim)
    axs[0].legend(loc='lower left')
    axs[0].set_xlabel('stimulus ' r"$\theta$")
    axs[0].set_ylim([-1,+1])
    axs[0].set_box_aspect(1/1)
    axs[0].set_title(r'Prior', fontsize=14, color='k');

    for k in range( Nneuron ):
        if k == 0:
            axs[1].plot( Theta_i, lnL_ki_ML[k,:], '-', linewidth=0.5, label=r"$\partial \ln L_k(\theta|x_k)/\partial \theta$")
        else:
            axs[1].plot( Theta_i, lnL_ki_ML[k,:], '-', linewidth=0.5 )
    axs[1].plot( Theta_i, lnL_i_ML, 'r-', linewidth=2, label="$\Sigma_k \partial\ln L_k() / \partial\theta$" )
    axs[1].plot( [Theta_true,Theta_true], YLim, 'r:', linewidth=2, label=r"$\theta_\mathit{true}$" )
    axs[1].plot( [Theta_prior,Theta_prior], [-1,+1], 'm:', linewidth=2, label=r"$\theta_\mathit{prior}$" )
    axs[1].plot( XLim, [0,0], 'k-', linewidth=0.5 )

    axs[1].set_xlim(XLim)
    axs[1].legend(loc='lower left')
    axs[1].set_xlabel('stimulus ' r"$\theta$")
    axs[1].set_ylim(YLim)
    axs[1].set_box_aspect(1/1)
    axs[1].set_title('Neurons', fontsize=14, color='k');

    axs[2].plot( Theta_i, lnP_i_prior, 'm-', linewidth=2, label='Prior')
    axs[2].plot( Theta_i, lnL_i_ML, 'r-', linewidth=2, label='ML' )
    axs[2].plot( Theta_i, lnL_i_MAP, 'k-', linewidth=2, label='MAP' )
    axs[2].plot( [Theta_true,Theta_true], YLim, 'r:', linewidth=2, label=r"$\theta_\mathit{true}$")
    axs[2].plot( [Theta_prior,Theta_prior], YLim, 'm:', linewidth=2, label=r"$\theta_\mathit{prior}$" )
    axs[2].plot( XLim, [0,0], 'k-', linewidth=0.5)

    axs[2].set_xlim(XLim)
    axs[2].legend(loc='lower left')
    axs[2].set_xlabel('stimulus ' r"$\theta$")
    axs[2].set_ylim(YLim)
    axs[2].set_box_aspect(1/1)
    axs[2].set_title('Combined', fontsize=14, color='k');

    plt.tight_layout()
    plt.show()

# Creating sliders for concentrations and charge
interact(plot_multiple_contributions_to_MAP, Sigma_prior = (0.2, 1.5, 0.1), Sigma_tun = (0.2, 1.5, 0.1), Nneuron = (10,100,10), toggle = (0, 1, 1) )


<>:85: SyntaxWarning: invalid escape sequence '\S'
<>:85: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipython-input-4115804110.py:85: SyntaxWarning: invalid escape sequence '\S'
  axs[1].plot( Theta_i, lnL_i_ML, 'r-', linewidth=2, label="$\Sigma_k \partial\ln L_k() / \partial\theta$" )


interactive(children=(FloatSlider(value=0.4, description='Sigma_prior', max=1.5, min=0.2), FloatSlider(value=1…

<unknown>:151: SyntaxWarning: invalid escape sequence '\S'


<function __main__.plot_multiple_contributions_to_MAP(Sigma_prior=0.4, Sigma_tun=1.0, Nneuron=20, toggle=0)>

# Solving the simplified MAP condition

If neurons $k$ cover possible stimuli $\theta$ uniformly and symmetrically, we can assume that

$\begin{eqnarray}
\sum_{k=1}^n \, r_k(\theta)  \left[\theta -\theta_k^\mathit{pref}\right] \approx 0 \qquad \forall \theta
\end{eqnarray}$

<br>

so that the MAP condition becomes

$\begin{eqnarray}
\\
0 &=&  -\frac{\theta - \theta_{prior}}{\sigma_{prior}^2}  -  \frac{\theta}{\sigma_{tuning}^2} \sum_{k=1}^n \, x_k - \frac{1}{\sigma_{tuning}^2} \, \sum_{k=1}^n x_k \, \theta_k^\mathit{pref}
\end{eqnarray}$

$\begin{eqnarray}
\\
\Leftrightarrow
\end{eqnarray}$

$\begin{eqnarray}
\\
\theta \, \left[  \frac{1}{\sigma_{tuning}^2} \, \sum_{k=1}^n \, x_k + \frac{1}{\sigma_{prior}^2} \right] = \frac{1}{\sigma_{tuning}^2} \, \sum_{k=1}^n \, x_k^\mathit{pref} \, \theta_k + \frac{1}{\sigma_{prior}^2}\theta_{prior}
\end{eqnarray}$

$\begin{eqnarray}
\\
\Leftrightarrow
\end{eqnarray}$

$\begin{eqnarray}
\\
\theta_{MAP}  &=& \frac{ \frac{1}{\sigma_{tuning}^2} \, \sum_{k=1}^n \, x_k \, \theta_k^\mathit{pref} + \frac{1}{\sigma_{prior}^2}\theta_{prior}}{\frac{1}{\sigma_{tuning}^2} \, \sum_{k=1}^n \, x_k^\mathit{pref} + \frac{1}{\sigma_{prior}^2}}
\end{eqnarray}$

(Slide-13)

# MAP estimate

The MAP estimate is

$\begin{eqnarray}
\\
\theta_{MAP}  &=& \frac{ \frac{1}{\sigma_{tuning}^2} \, \sum_{k=1}^n \, x_k \, \theta_k^\mathit{pref} + \frac{1}{\sigma_{prior}^2}\theta_{prior}}{\frac{1}{\sigma_{tuning}^2} \, \sum_{k=1}^n \, x_k^\mathit{pref} + \frac{1}{\sigma_{prior}^2}}
\end{eqnarray}$

<br>

a response-weighted average over neurons -- preferred stimuli $\theta_k^\mathit{pref}$ -- and the most probable prior stimulus $s_{prior}$.

Both sources of information -- neurons and prior -- are weighted by **inverse variance**.

**In Bayesian inference, estimates from different sources of information are generally weighted by the inverse variance.**

(Slide-14)

# Summary MAP inference


- The MAP estimate $\theta_\mathit{MAP}$ is an average of an average.

- The first average is over the preferred stimuli $\theta_k^\mathit{pref}$ of the neurons and is weighted by neurons.

- The second average is over the neural estimate and teh most probable prior stimulus $\theta_{prior}$

- The respective weights are inverse variance:  $1/\sigma_\mathit{tuning}^2$ and  $1/\sigma_\mathit{prior}^2$.

- We have illustrated graphically the solution of the MAP condition for multiple responses.

(Slide-15)



# **3. Computing log-likelihood with feedforward projections**


Bayesian formalisms let us optimally combine sensory information from different sources (e.g., diverse neurons responding to the same stimulus, or non-uniform prior probabilities).

Bayesian formalisms require that all sources are well characterized, in terms of how they respond to different stimuli.

Once these prior characterizations are available, the remaining computations are not complex and readily performed by neurons: weighted sums and peak localization.


Following [Jazayeri \& Movshon (2006)](https://www.nature.com/articles/nn1691), we now compute log-likelihood with feedforward projections from a population of **sensory neurons** to a population of **log-likelihood neurons**.

(Slide-16)

# Sensory population

Consider a population of sensory neurons $i$ with circular Gaussian tuning $f_i(\theta)$, Poisson variability, uniform and symmetric coverage, and independently variable responses $n_i$:

$\begin{eqnarray}
\\
f_i(\theta) = r_{max}\, e^{\left[ \kappa \, \cos(\theta-\theta_i^\mathit{pref}) - \kappa \right]}
\end{eqnarray}$

$\begin{eqnarray}
\\
\ln p(n_i|\theta) \quad \approx  \quad n_i \ln f_i(\theta) \quad \approx \quad  n_i \, \cos(\theta-\theta_i)
\end{eqnarray}$

<br>

'Circular' Gaussian tuning is nearly the same as 'ordinary' Gaussian tuning but analytically (even) more convenient!

<br>

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec14_Illu02.png" width="800">

(Slide-17)

# Only one stimulus-dependent term

Applying ML inference to the responses of the sensory population, only one term of the joint log likelihood depends on the stimulus (i.e., varies with $\theta$)

$\begin{eqnarray}
\ln L(\theta|n_{1:N}) &=& \sum_{i=1}^N \, \ln P(n_i|\theta)  =
\\
\\
&\approx & \sum_{i=1}^N \, n_i \, \ln f_i(\theta) =
\\
\\
 &\approx & \sum_{i=1}^N \, n_i \, \cos(\theta - \theta_i^\mathit{pref})
\end{eqnarray}$

The one remaining term is a sum of products: *observed* responses $\times$ the logarithm of *expected* response.


The ML estimate is the value $\theta_{ML}$ that maximises this sum of products.

(Slide-18)

# Likelihood population

To represent $\ln L(\theta|n_{1:N})$ with neuronal activity, we construct a **likelihood population** of neurons $j$ with responses $L_j$ and preferred stimuli $\theta_j^\mathit{pref}$.   The activity of second layer neurons is obtained in a feedforward fashion as weighted sums of **sensory responses** $n_i$.

$\begin{eqnarray}
\qquad\qquad L_j = \sum_{i=1}^N \, n_i \, g_{ij}
\end{eqnarray}$

where $g_{ij}$ is the 'synaptic weight' of the projection from sensory neuron $i$ to likelihood neuron $j$.

In the summer semester, we will justify this simplistic treatment of neuronal projections.

<br>

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec15_Likelihood_population.png" width="400">

(Slide-19)

# Choosing synaptic weights wisely

We choose synaptic weights that are shift-invariant

$\begin{eqnarray}
g_{ij} = g(\theta_i^\mathit{pref} - \theta_j^\mathit{pref})
\end{eqnarray}$

in that they depend only on the *differential* stimulus preference at the two levels. Specifically, we choose the weights proportional to the **logarithm of the expected response** of presynaptic neuron $i$ to the preferred stimulus of postsynaptic neuron $j$.

$\begin{eqnarray}
g_{ij} = g(\theta_i^\mathit{pref} - \theta_j^\mathit{pref}) \approx \ln f_i(\theta_j^\mathit{pref}) = \cos\left(\theta_i^\mathit{pref} - \theta_j^\mathit{pref} \right)
\end{eqnarray}$

With this choice, the activity of postsynaptic neuron $j$

$\begin{eqnarray}
L_j = \sum_{i=1}^N \, n_i \, g_{ij} =  \sum_{i=1}^N \, n_i \, \cos(\theta_i - \theta_j)  = \ln L(\theta_j|n_{1:N})
\end{eqnarray}$

becomes proportional to the joint log-likelihood of its preferred stimulus $\theta_j^\mathit{pref}$, given the activities $n_{1:N}$ of sensory neurons $i$. This is what makes it a **likelihood neuron**.

In short, likelihood neurons give *positive* weight to sensory neurons with the *same* preference, $\theta_i^\mathit{pref} =\theta_j^\mathit{pref}$, and *negative* weight to sensory neurons with *different* preference, $\theta_i^\mathit{pref} \neq \theta_j^\mathit{pref}$.

(Slide-20)

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec15_JM01.png" width="800">

(Slide-21)

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec15_JM02.png" width="800">

(Slide-22)

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec15_JM03.png" width="800">

(Slide-23)

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec15_JM04.png" width="800">

(Slide-24)

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec15_JM05.png" width="800">

(Slide-25)

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec15_JM06.png" width="800">

(Slide-26)

# Summary computing likelihood with feedforward projections



- In visual cortical area V1, neurons $i$ are tuned for directions of visual motion $\theta_i$ and their response variability is Poisson distributed.


- Jazayeri and Movshon (2006) proposed that neurons in the visual cortical area MT perform log-likelihood decoding by summing over the V1 activities $x_i^{V1}$ with suitable weights $w_{ji}$.

$\begin{eqnarray}
\qquad\qquad x_j^{MT} = \sum_{i=1}^n \, w_{ji} \, x_i^{V1}
\end{eqnarray}$
  
- Specifically, they propose that an area MT neuron representing $\theta_j$ applies weights proportional to

$\begin{eqnarray}
\qquad\qquad w_{ji} = \log f_i(\theta_j) \approx \cos\left(\theta_j - \theta_i\right)
\end{eqnarray}$


- In this way, MT activity $x_j^{MT}$ approximates the log-likelihood that $\theta_j$ caused V1 activity $x_{1:n}^{V1}$.

- This scenario assumes that visual development and plasticity is able to converge to this particular pattern of projection weights. In the summer semester, we will encounter plausible mechanisms for this (Hebbian plasticity).

(Slide-27)


# **4. Discriminating, detecting, and identifying with feedforward projections**

The decoding scheme proposed by Jazayeri \& Movshon (2006) works equally well for

"Identification tasks" (stimulus A, B, C, ... )

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec12_task_identification.png" width="100">

"Discrimination tasks" (stimulus A or B?)

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec12_Task_discrimination.png" width="100">


"Detection tasks" (stimulus A absent or present?)

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec12_Task_detection.png" width="100">

(Slide-28)






<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec15_JM07.png" width="800">

(Slide-29)

# Differential contribution of sensory subpopulations

It is instructive to consider the differential contribution of different sensory subpopulations for different kinds of tasks.

# Which subpopulations contribute to a coarse discrimination?

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec15_JM08.png" width="800">

(Slide-30)

# Those that are tuned best!


<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec15_JM09.png" width="800">

(Slide-31)

# Which subpopulations contributed to a "fine discrimination"?

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec15_JM10.png" width="800">

(Slide-32)

# Neither the fully tuned nor the untuned!

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec15_JM12.png" width="800">

(Slide-33)

# Those that are half-tuned!

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec15_JM13.png" width="800">

(Slide-34)

# Quantify intuition for coarse discrimination

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec15_JM14.png" width="800">

(Slide-35)

# Quantify intuition for fine discrimination

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec15_JM15.png" width="800">

(Slide-36)

# Half-tuning crucial for fine discrimination

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec15_JM16.png" width="800">

(Slide-37)

# Informativeness for fine discrimination


For fine discrimination, the most informative subpopulations are half-tuned rather than fully-tuned.

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec15_JM17.png" width="400">

(Slide-38)

# Summary of ML inference with feedforward projections

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec15_JM18.png" width="800">

(Slide-39)

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Lec15_JM_Figure_1.png" width="600">

At its input (bottom), a stimulus, elicits $n_1$, $n_2$, ..., $n_N$ spikes in the sensory representation. The response of each neuron multiplied by the logarithm of its own tuning curve, $\log[f_i]$, gives the contribution of that neuron to the log likelihood function. Adding the contribution of individual neurons (shown for two example stimulus values in orange and green) gives the overall log likelihood function, $\log L(\theta)$ for all values of $\theta$ that could have elicited this pattern of responses. Here, the orange point at the peak of the log likelihood function indicates the most likely stimulus.

(Slide-39)

# **What you should know about population decoding**

- To interpret neuronal responses efficiently, we need a full characterization of the responsiveness of each neuron in the population.

<br>

- Specifically, we need to know the mean response to different stimuli (**tuning curve**) and the distribution of actual responses around this mean (**response variability**).

<br>

- Taken together, this provides the conditional probability $P(x|s)$ of different possible responses $x$, given different possible stimuli $s$.

<br>

- If responses vary independently, the joint probability is the product of the individual probabilities

$\begin{eqnarray}
\qquad\qquad P(x_{1:n}|s) = \prod_{i=1}^n \, P(x_i|s)
\end{eqnarray}$

<br>

- The stimulus $s_\mathit{ML} that maximizes the joint probability is the **maximum likelihood (ML)** estimate.

<br>

- $L(s|x)$ is the **posterior likelihood** of different possible stimuli $s$, given different observed responses $x$.  

<br>

- $L(s|x)$ differs from $P(x|s)$ by taking into account a non-uniform **prior probability** $P(s)$ of stimuli $s$

$\begin{eqnarray}
\qquad\qquad L(s|x)  \approx P(x|s) \, P(s)
\end{eqnarray}$

<br>

- If responses vary independently, the joint likelihood is the product of the individual likelihoods

$\begin{eqnarray}
\qquad\qquad L(s| x_{1:n}) = \prod_{i=1}^n \, L(s| x_i)
\end{eqnarray}$

<br>

- The stimulus $s_\mathit{MAP}$ maximizing the joint likelihood is the **maximum a posteriori (MAP)** estimate.  


(Slide-40)

# **After Xmas:**

# **16. Shannon information**

# **17. Mutual information**

# **18. Neural applications**
